# 06 — a100res1024_s42 numa L4

Este notebook roda **um braco no Colab Pro+**, com o Drive como arquivo. E o caminho para treinar em paralelo com o Colab Enterprise: a cota de GPU do projeto GCP vale para o projeto inteiro (1 GPU), e a L4 da assinatura nao passa por ela.

A fila:

1. `configs/allsky/experiments/l4/a100res1024_s42.yaml`

Os configs vivem no repositorio e sao rodados como estao: este notebook nao escreve YAML
derivado. O `amp` deles e `bf16` (a L4 e Ada); a celula de hardware confere que a GPU
atribuida tem bfloat16 antes de baixar 1,4 GB de dados.

## O que garante que nada se perde

Uma VM do Colab Enterprise e devolvida sem aviso e morre junto com o notebook: o que estiver
so no disco dela quando algo levantar excecao esta perdido, e numa fila de treino isso e
hora de GPU. Cinco coisas seguram isso, e todas tem teste em `tests/allsky/test_colab_runner.py`:

1. **Voo de teste antes da fila.** A secao 5 percorre, com dados sinteticos e em menos de um
   minuto, cada passo que roda fora do treino — pontuar por bloco no interpretador do venv,
   arquivar, espelhar ao vivo, retomar e escrever no destino final. Um `import` que so existe
   no venv, um caminho sem permissao ou um espelho apontando para lugar nenhum falham aqui,
   nao na decima quarta hora.
2. **Cada avaliacao e arquivada e espelhada assim que existe**, antes da proxima comecar e
   antes de qualquer pontuacao.
3. **Checkpoints viajam no espelho ao vivo** (`_live/<braco>/last.ckpt`), a cada 5 min. Uma
   execucao relancada restaura o checkpoint e continua com `allsky train --resume auto`.
4. **Nada depois do treino levanta.** Falha em pontuar vira coluna vazia e nota na linha;
   falha num braco nao derruba a fila.
5. **O relato sobrevive a sessao.** Cada passo vai para `fila_l4.log` no bucket.

## O que fica arquivado, e onde

Em `MyDrive/labmim/runs/allsky-l4-a100res1024_s42/` (o Drive e o arquivo: nao ha espelho a fazer, o que se escreve ali ja esta fora da VM):

- `<braco>/` — `metrics.csv`/`metrics.json` do treino, os relatorios `eval-test` (best),
  `eval-val` (best) e `eval-test-last` (last) com `predictions.parquet`, `stratified.csv`,
  `confusion.csv` e `report.md`, o YAML do config **e os checkpoints**.
- `_live/` — `metrics.csv`, `last.ckpt`, `best.ckpt` de cada braco e `heartbeat.json` com a
  GPU: e o que diz, de fora, se a VM esta treinando ou parada.
- `fila_l4.log`, `campanha_parcial.csv` a cada braco, `campanha.csv` e `campanha_resumo.json`
  no fechamento, `preflight.json` no comeco.

## Antes de rodar

1. **Runtime -> Change runtime type -> L4 GPU** (ou A100). Os configs declaram
   `amp: bf16`, entao uma T4 para na celula de hardware, antes de desempacotar 1,4 GB.
2. O Drive tem `MyDrive/labmim/allsky-mm/bundle-iso-20260906.tar.gz` e
   `MyDrive/labmim/dinov3/dinov3_vits16plus_pretrain_lvd1689m.pth`.
3. Rode as celulas em ordem. A de ambiente leva uns 8 min (venv 3.14 mais torch CUDA) e a
   fila de 8 a 10 h: deixe a aba aberta, com execucao em segundo plano ligada.
4. Para pular o braco com a sessao ja rodando: crie `MyDrive/labmim/fila-l4/<braco>.skip`.

## 1. Runtime e GPU

In [ ]:
import subprocess
import time

SESSION_START = time.time()
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True, check=False).stdout)

## 2. Ambiente

Clona o repositorio onde o `_colab_runner` mora, instala o torch CUDA pelo backend que o
driver da VM pede e verifica. O repositorio vem de um `git bundle` no Drive: a branch desta campanha e local e nunca foi publicada.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")
BRANCH = "condicao-do-ceu-multitarefa"
STORE = "/content/drive/MyDrive/labmim"
BASE = "/content"
WORKDIR = f"{BASE}/micrometeorology"

# O repositorio vem do git bundle no Drive, nao do GitHub: a branch desta campanha
# e local e nunca foi publicada.
BUNDLE_GIT = f"{STORE}/colab/micrometeorology.bundle"
if not os.path.exists(BUNDLE_GIT):
    raise RuntimeError(f"{BUNDLE_GIT} nao existe no Drive — suba o git bundle antes de rodar")
if not os.path.exists(WORKDIR):
    subprocess.run(["git", "clone", "-b", BRANCH, BUNDLE_GIT, WORKDIR], check=True)
if not os.path.isdir(f"{WORKDIR}/configs/allsky/experiments/l4"):
    raise RuntimeError(
        f"o bundle de {BRANCH} nao carrega configs/allsky/experiments/l4/ — refaca o bundle"
    )
subprocess.run(["pip", "install", "-q", "uv"], check=True)
subprocess.run(["uv", "python", "install", "3.14"], cwd=WORKDIR, check=True)
subprocess.run(["uv", "venv", "--python", "3.14", ".venv"], cwd=WORKDIR, check=True)
subprocess.run(["uv", "sync", "--locked", "--extra", "allsky"], cwd=WORKDIR, check=True)
subprocess.run(
    [
        "uv",
        "pip",
        "install",
        "--python",
        ".venv/bin/python",
        "--reinstall",
        "--torch-backend",
        "auto",
        "torch==2.13.0",
    ],
    cwd=WORKDIR,
    check=True,
)

PY = f"{WORKDIR}/.venv/bin/python"
os.environ["PATH"] = f"{WORKDIR}/.venv/bin:" + os.environ["PATH"]

DINOV3_REPO_DIR = f"{BASE}/dinov3"
if not os.path.exists(DINOV3_REPO_DIR):
    subprocess.run(
        [
            "git",
            "clone",
            "--depth",
            "1",
            "https://github.com/facebookresearch/dinov3",
            DINOV3_REPO_DIR,
        ],
        check=True,
    )
os.environ["ALLSKY_DINOV3_REPO"] = DINOV3_REPO_DIR
sys.path.insert(0, f"{WORKDIR}/notebooks/colab")

verify = subprocess.run(
    [PY, "-c", "import torch; print(torch.__version__, torch.cuda.is_available())"],
    capture_output=True,
    text=True,
    check=False,
)
print(verify.stdout)
if "True" not in verify.stdout:
    raise RuntimeError(
        "torch sem CUDA — Runtime > Change runtime type > GPU, e rode esta celula de novo"
    )

import _colab_runner as runner  # noqa: E402

lacking = [
    name
    for name in ("preflight", "run_arm", "pull_live_run", "score_by_sensor_block_in")
    if not hasattr(runner, name)
]
if lacking:
    raise RuntimeError(
        f"o _colab_runner de {BRANCH} nao tem {lacking} — refaca o bundle de uma branch que os carregue"
    )

## 3. Hardware

O probe roda no interpretador do venv. Os configs declaram `amp: bf16`, entao uma GPU sem
bfloat16 (T4, Turing) para aqui — antes de baixar o bundle de 1,4 GB.

In [ ]:
import json

probe = subprocess.run(
    [
        PY,
        "-c",
        'import json, sys; sys.path.insert(0, "'
        + WORKDIR
        + '/notebooks/colab"); import _colab_runner as r; print(json.dumps(r.probe_accelerator()))',
    ],
    capture_output=True,
    text=True,
    check=True,
)
HW = json.loads(probe.stdout.strip().splitlines()[-1])
print(HW)
if HW["amp_dtype"] != "bf16":
    raise RuntimeError(
        f"{HW['name']} sem bfloat16: os configs de l4/ declaram amp bf16 — peca o template labmim-l4"
    )
if HW["cpus"] < 8:
    print(f"ATENCAO: {HW['cpus']} vCPU para num_workers: 8 dos configs — o loader vai disputar CPU")

## 4. Dados e artefatos

So bucket. O que sai da VM sai por `ARTIFACTS`, espelhado para o prefixo deste notebook;
`fila-l4/` e lido do bucket antes de cada braco e **nunca** espelhado de volta, porque o `rsync`
nao tem direcao e a copia antiga da VM sobrescreveria o arquivo posto la de fora.

In [ ]:
BUNDLE = f"{STORE}/allsky-mm/bundle-dataset-iso-1024-20260910.tar.gz"
WEIGHTS = f"{STORE}/dinov3/dinov3_vits16plus_pretrain_lvd1689m.pth"
for caminho in (BUNDLE, WEIGHTS):
    if not os.path.exists(caminho):
        raise RuntimeError(f"{caminho} nao existe no Drive — suba antes de rodar")
os.environ["ALLSKY_DINOV3_WEIGHTS"] = WEIGHTS

DATA = "/content/allsky-mm"
ARTIFACTS = f"{STORE}/runs/allsky-l4-a100res1024_s42"
OVERRIDES = f"{STORE}/fila-l4"
REMOTE_OVERRIDES = None
os.makedirs(ARTIFACTS, exist_ok=True)
os.makedirs(OVERRIDES, exist_ok=True)
# O Drive montado e o proprio arquivo: o que se escreve ali ja esta fora da VM,
# entao nao ha destino remoto a espelhar.
MIRROR = []
print("ja arquivado:", sorted(p.name for p in Path(ARTIFACTS).iterdir()) or "nada")

ROOT = runner.stage_bundle(BUNDLE, DATA, python=PY)
for required in ("manifest.parquet", "splits.json", "frames"):
    if not (Path(ROOT) / required).exists():
        raise RuntimeError(
            f"{ROOT} sem {required}: o bundle nao e o dataset-iso-1024-20260910 com frames"
        )

DATASET_LINK = Path(WORKDIR) / "output/allsky-mm/dataset-iso-1024-20260910"
DATASET_LINK.parent.mkdir(parents=True, exist_ok=True)
if DATASET_LINK.is_symlink():
    DATASET_LINK.unlink()
elif DATASET_LINK.exists():
    raise RuntimeError(f"{DATASET_LINK} existe e nao e um link: nao vou sobrescrever")
DATASET_LINK.symlink_to(ROOT, target_is_directory=True)
os.chdir(WORKDIR)
OUT = Path(WORKDIR) / "output/allsky-mm/experiments/l4"
OUT.mkdir(parents=True, exist_ok=True)
print(f"{DATASET_LINK} -> {os.readlink(DATASET_LINK)}; cwd {os.getcwd()}; runs em {OUT}")

## 5. Voo de teste

Percorre, com dados sinteticos, cada passo que roda fora do treino, e escreve
`preflight.json` no destino final. E o que transforma uma quebra de quatorze horas numa de um
minuto. Em seguida arma o espelho ao vivo (checkpoints inclusos) e o espelho para o bucket.

In [ ]:
for check in runner.preflight(PY, artifacts=ARTIFACTS, mirror=MIRROR, work_dir=BASE):
    print("  ok:", check)
WATCHERS = [
    runner.start_live_sync(OUT, Path(ARTIFACTS) / runner.LIVE_DIR),
    runner.start_mirror(MIRROR),
]

## 6. A fila

Cada entrada passa por `runner.run_arm`, que tem teste: override do bucket, retomada do
checkpoint espelhado, treino, as tres avaliacoes **arquivando e espelhando cada uma assim que
existe**, e so entao a pontuacao por bloco, no interpretador do venv e dentro de um `try`.
Nada aqui levanta: o estado de cada braco vira uma linha da tabela.

In [ ]:
from pathlib import Path

import pandas as pd

FILA = ["configs/allsky/experiments/l4/a100res1024_s42.yaml"]
missing = [rel for rel in FILA if not (Path(WORKDIR) / rel).is_file()]
if missing:
    raise RuntimeError(f"{missing}: a branch {BRANCH} do bundle nao carrega os configs de l4/")

LOG = Path(ARTIFACTS) / "fila_l4.log"
rows = []


def log(line):
    """Print *line* and append it, timestamped, to the log the mirror carries."""
    stamped = f"[{time.strftime('%Y-%m-%dT%H:%M:%S%z')}] {line}"
    print(stamped)
    with open(LOG, "a", encoding="utf-8") as handle:
        handle.write(stamped + "\n")


for entry in FILA:
    started = time.time()
    row = runner.run_arm(
        Path(WORKDIR) / entry,
        python=PY,
        out_dir=OUT,
        artifacts=Path(ARTIFACTS),
        mirror=MIRROR,
        overrides=Path(OVERRIDES),
        override_mirror=[(REMOTE_OVERRIDES, OVERRIDES)] if REMOTE_OVERRIDES else [],
        watchers=WATCHERS,
        log=log,
    )
    rows.append(row)
    pd.DataFrame(rows).to_csv(f"{ARTIFACTS}/campanha_parcial.csv", index=False)
    log(runner.summarise_arm(row))
    log(
        f"  {(time.time() - started) / 3600:.1f} h nesta entrada; {(time.time() - SESSION_START) / 3600:.1f} h de sessao"
    )
    runner.mirror_once(MIRROR)

## 7. Fechamento

Grava o indice da campanha e espelha uma ultima vez.

In [ ]:
frame = pd.DataFrame(rows)
frame.to_csv(f"{ARTIFACTS}/campanha.csv", index=False)
with open(f"{ARTIFACTS}/campanha_resumo.json", "w") as handle:
    json.dump(
        {
            "hardware": HW,
            "fila": FILA,
            "n_runs": len(rows),
            "session_hours": round((time.time() - SESSION_START) / 3600, 2),
            "rows": rows,
        },
        handle,
        indent=2,
        default=str,
    )
for row in rows:
    print(runner.summarise_arm(row))
print(frame.to_string())
if MIRROR:
    print("espelho final:", runner.mirror_once(MIRROR) or "ok")
print("artefatos em", ARTIFACTS)